# 06 - Generation And API (Hue Foods RAG MVP)

Notebook này chạy **full runtime path thật** của Phase 6 qua API: câu hỏi -> retrieval thật (Qdrant + E5) -> ContextBuilder thật -> OpenAI generator thật (`gpt-5.4-nano` qua OpenAI Agents SDK) -> answer-only response.

**Phase 6 - Simplicity và Single-Turn API**

- FastAPI lifespan build retrieval stack một lần và lưu readiness trong `app.state`.
- Request `/api/chat` nhận `query` và trả JSON chỉ gồm `{"answer": "..."}`.
- Backend log các bước chính ra console và `backend/logs/application.log`; log và exception không đi vào response.

**Prerequisite**

- Khởi động Jupyter từ terminal đã export `OPENAI_API_KEY` vào environment (notebook không đọc `.env`, không dùng `load_dotenv`). Nếu thiếu key, notebook fail actionable ngay - không có fallback.
- Qdrant local đang chạy (collection `hue_foods_e5_small_384`, 572 points) và E5 đã cache.

**Chi phí**

- Mỗi Run All gọi **đúng 1** OpenAI call (`POST /api/chat`). Chi phí ước tính dưới 0,001 USD mỗi call.

**Kết quả mong đợi khi Run All**

- `/health` trả `ok` với các component ready.
- `/api/chat` trả HTTP 200: JSON response có đúng một trường `answer` với câu trả lời tiếng Việt grounded.
- Backend log ghi nhận đầy đủ câu hỏi, retrieval và generation tại server console/file.


In [2]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


backend on path: /home/minhhieu/hue_rag/backend


## Kiểm tra key

Cell dưới chỉ kiểm tra **presence** của `OPENAI_API_KEY` (không in giá trị). Nếu key thiếu, notebook dừng với thông báo rõ ràng.


In [3]:
import os

has_key = bool(os.environ.get("OPENAI_API_KEY", "").strip())
print("OPENAI_API_KEY present:", has_key)
if not has_key:
    raise RuntimeError(
        "Thieu OPENAI_API_KEY trong environment. Hay khoi dong Jupyter tu "
        "terminal da export key (export OPENAI_API_KEY=...) roi chay lai."
    )


OPENAI_API_KEY present: True


## Câu hỏi

Người dùng chỉ cần sửa đúng một biến dưới đây rồi Run All. API tự chạy retrieval, build labeled context và sinh câu trả lời.


In [4]:
question = "Bún bò Huế có đặc điểm gì nổi bật?"
print("question:", question)


question: Bún bò Huế có đặc điểm gì nổi bật?


## Gọi `/api/chat` qua app thật

Cell dưới dùng FastAPI `TestClient` với app thật và lifespan thật: lifespan build retrieval stack một lần, `/health` đọc cached readiness và `POST /api/chat` chạy retrieval + OpenAI generation.

Success response chỉ có một trường `answer`. Backend log các giai đoạn ra server console và file log, không đưa vào API response.


In [5]:
from fastapi.testclient import TestClient

from api.app import app

with TestClient(app) as client:
    health = client.get("/health")
    response = client.post("/api/chat", json={"query": question})

print("health:", health.json())
print("status:", response.status_code)
print("response fields:", list(response.json()))
print("answer:", response.json()["answer"])


/home/minhhieu/hue_rag/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


[2026-08-26 09:06:07] INFO - sentence_transformers.base.model - Loading SentenceTransformer model from intfloat/multilingual-e5-small.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/minhhieu/hue_rag/backend/embedding/embedder.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  actual_dimension = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-08-26 09:06:27] INFO - chat - Received question: Bún bò Huế có đặc điểm gì nổi bật?
[2026-08-26 09:06:27] INFO - chat - Running retrieval


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-08-26 09:06:27] INFO - retrieval.service - retrieval profile=dense_only documents=10
[2026-08-26 09:06:27] INFO - chat - Retrieved 10 documents
[2026-08-26 09:06:27] INFO - llm - Generating answer with model: gpt-5.4-nano
[2026-08-26 09:06:31] INFO - llm - Generated answer successfully in 4157 ms; tokens=1063/179
[2026-08-26 09:06:31] INFO - chat - Chat request completed successfully
health: {'status': 'ok', 'components': {'app': 'alive', 'qdrant': 'ready', 'retrieval': 'ready', 'generator': 'configured'}}
status: 200
response fields: ['answer']
answer: Bún bò Huế nổi bật bởi nét mộc mạc dân gian nhưng vẫn tinh tế, gắn chặt với văn hóa ẩm thực Huế. Về hương vị, món có nước dùng đậm đà từ xương, sả và mắm ruốc; kết hợp với sợi bún và các thành phần như thịt bò, giò heo, huyết luộc, chả cua/chả bò cùng rau ăn kèm. Khi nhắc “bún bò Huế” thường là để nhấn mạnh xuất xứ và phong cách chế biến đặc trưng; ở Huế nước dùng thiên về vị thanh, đậm mùi mắm ruốc và sả, có vị cay rõ và thường ă

In [6]:
print(response.json()["answer"])

Bún bò Huế nổi bật bởi nét mộc mạc dân gian nhưng vẫn tinh tế, gắn chặt với văn hóa ẩm thực Huế. Về hương vị, món có nước dùng đậm đà từ xương, sả và mắm ruốc; kết hợp với sợi bún và các thành phần như thịt bò, giò heo, huyết luộc, chả cua/chả bò cùng rau ăn kèm. Khi nhắc “bún bò Huế” thường là để nhấn mạnh xuất xứ và phong cách chế biến đặc trưng; ở Huế nước dùng thiên về vị thanh, đậm mùi mắm ruốc và sả, có vị cay rõ và thường ăn kèm nước mắm dầm ớt tươi.


## Checklist xác nhận Phase 6

1. `OPENAI_API_KEY present: True` được in ra (không bao giờ in giá trị key).
2. `/health` trả `ok` với `qdrant: ready`, `retrieval: ready`, `generator: configured`.
3. `/api/chat` trả HTTP 200 với đúng một trường `answer` tiếng Việt grounded.
4. Response fields chỉ có `["answer"]`; không chứa `sources`, `session_id`, `retrieval_debug` hay backend logs.
5. Đúng 1 OpenAI call cho mỗi Run All.
6. Backend log hiển thị câu hỏi, retrieval và generation tại server console/file log.
